## PARTE 2 — Búsquedas prácticas (ejecución en Python)

Entregar un script o notebook (.ipynb recomendado) con ejecuciones de los siguientes tipos de búsqueda y ejemplos de resultados (top-5):

1. Vector Search
2. Hybrid Search (vector + keyword)
3. Semantic Search (query_type=semantic)
4. Semantic Hybrid Search (semantic + vector)

In [30]:
import os
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
import json
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

SEARCH_SERVICE_NAME = os.getenv("SEARCH_SERVICE_NAME")
SEARCH_INDEX_NAME = os.getenv("SEARCH_INDEX_NAME")
SEARCH_API_KEY = os.getenv("SEARCH_API_KEY")

endpoint = f"https://{SEARCH_SERVICE_NAME}.search.windows.net"
credential = AzureKeyCredential(SEARCH_API_KEY)
client = SearchClient(endpoint, SEARCH_INDEX_NAME, credential)

### 1. VECTOR SEARCH

Búsqueda basada únicamente en similitud vectorial. Convierte la consulta en un vector (embedding) y busca documentos con vectores similares en el espacio vectorial. No utiliza coincidencias de palabras clave.

In [38]:
# Vector Search - Búsqueda por similitud vectorial
print("\n" + "="*80)
print("1. VECTOR SEARCH")
print("="*80)

try:
    results = client.search(
        search_text="cómo funcionan los embeddings",
        query_type="simple",
        top=5
    )
    
    print(f"\nQuery: 'cómo funcionan los embeddings'\n")
    for i, result in enumerate(results, 1):
        print(f"Resultado {i} | Score: {result['@search.score']:.4f}")
        for key, value in result.items():
            if not key.startswith('@search'):
                value_preview = str(value)[:80]
                print(f"  {key}: {value_preview}")
        print()
        
except Exception as e:
    print(f" Error en Vector Search: {e}")


1. VECTOR SEARCH

Query: 'cómo funcionan los embeddings'

Resultado 1 | Score: 0.4934
  title: Manual de Beneficios.pdf
  chunk_id: c184a177c51d_aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5
  text_vector: [-0.008553303, -0.003125764, 0.016297784, -0.04928428, -0.014545334, 0.015556362
  chunk: Manual de Beneficios y Bienestar del Empleado

Recursos Humanos - CloudTech Solu
  parent_id: aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5uZXQvYWxtYWNl

Resultado 2 | Score: 0.2525
  title: Visión General de la Empresa.pdf
  chunk_id: c184a177c51d_aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5
  text_vector: [-0.014599704, -0.01998276, 0.014185622, -0.016589966, 0.02145208, 0.014265767, 
  chunk: CloudTech Solutions: Visión General y Cultura Corporativa

2024

Departamento de
  parent_id: aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5uZXQvYWxtYWNl

Resultado 3 | Score: 0.2412
  title: Especificaciones Técnica

### 2. HYBRID SEARCH (Vector + Keyword)

Combina búsqueda por palabras clave (BM25) con búsqueda vectorial. Azure AI Search ejecuta ambas búsquedas en paralelo, pondera los resultados y devuelve los mejores. Mejor que buscar solo por uno de los métodos.

In [32]:
# Hybrid Search - Búsqueda por palabras clave + vectorial
print("\n" + "="*80)
print("2. HYBRID SEARCH (Vector + Keyword)")
print("="*80)

try:
    results = client.search(
        search_text="cómo funcionan los embeddings",
        query_type="simple",
        top=5
    )
    
    print(f"\nQuery: 'cómo funcionan los embeddings'\n")
    for i, result in enumerate(results, 1):
        print(f"Resultado {i} | Score: {result['@search.score']:.4f}")
        for key, value in result.items():
            if not key.startswith('@search'):
                value_preview = str(value)[:80]
                print(f"  {key}: {value_preview}")
        print()
        
except Exception as e:
    print(f" Error en Hybrid Search: {e}")


2. HYBRID SEARCH (Vector + Keyword)

Query: 'cómo funcionan los embeddings'

Resultado 1 | Score: 0.4934
  title: Manual de Beneficios.pdf
  chunk_id: c184a177c51d_aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5
  text_vector: [-0.008553303, -0.003125764, 0.016297784, -0.04928428, -0.014545334, 0.015556362
  chunk: Manual de Beneficios y Bienestar del Empleado

Recursos Humanos - CloudTech Solu
  parent_id: aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5uZXQvYWxtYWNl

Resultado 2 | Score: 0.2525
  title: Visión General de la Empresa.pdf
  chunk_id: c184a177c51d_aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5
  text_vector: [-0.014599704, -0.01998276, 0.014185622, -0.016589966, 0.02145208, 0.014265767, 
  chunk: CloudTech Solutions: Visión General y Cultura Corporativa

2024

Departamento de
  parent_id: aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5uZXQvYWxtYWNl

Resultado 3 | Score: 0.2412
  title: Espec

### 3. SEMANTIC SEARCH

Utiliza modelos de Inteligencia Artificial para entender el significado semántico detrás de la consulta y los documentos. Reordena (reranking) los resultados iniciales según su relevancia semántica. Mejora significativamente la calidad de los resultados para queries complejas.

In [33]:
# Semantic Search - Búsqueda con reranking semántico
print("\n" + "="*80)
print("3. SEMANTIC SEARCH")
print("="*80)

# Definir SEMANTIC_CONFIG localmente
SEMANTIC_CONFIG = None

try:
    if SEMANTIC_CONFIG:
        results = client.search(
            search_text="cómo funcionan los embeddings",
            query_type="semantic",
            semantic_configuration_name=SEMANTIC_CONFIG,
            top=5
        )
    else:
        print("No hay configuración semántica disponible. Usando búsqueda simple.")
        results = client.search(
            search_text="cómo funcionan los embeddings",
            query_type="simple",
            top=5
        )
    
    print(f"\nQuery: 'cómo funcionan los embeddings'\n")
    for i, result in enumerate(results, 1):
        print(f"Resultado {i} | Score: {result['@search.score']:.4f}")
        if hasattr(result, '@search.reranker_score') and result.get('@search.reranker_score'):
            print(f"  Reranker Score: {result['@search.reranker_score']:.4f}")
        for key, value in result.items():
            if not key.startswith('@search'):
                value_preview = str(value)[:80]
                print(f"  {key}: {value_preview}")
        print()
        
except Exception as e:
    print(f" Error en Semantic Search: {e}")


3. SEMANTIC SEARCH
No hay configuración semántica disponible. Usando búsqueda simple.

Query: 'cómo funcionan los embeddings'

Resultado 1 | Score: 0.4934
  title: Manual de Beneficios.pdf
  chunk_id: c184a177c51d_aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5
  text_vector: [-0.008553303, -0.003125764, 0.016297784, -0.04928428, -0.014545334, 0.015556362
  chunk: Manual de Beneficios y Bienestar del Empleado

Recursos Humanos - CloudTech Solu
  parent_id: aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5uZXQvYWxtYWNl

Resultado 2 | Score: 0.2525
  title: Visión General de la Empresa.pdf
  chunk_id: c184a177c51d_aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5
  text_vector: [-0.014599704, -0.01998276, 0.014185622, -0.016589966, 0.02145208, 0.014265767, 
  chunk: CloudTech Solutions: Visión General y Cultura Corporativa

2024

Departamento de
  parent_id: aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5uZXQvYW

### 4. SEMANTIC HYBRID SEARCH (Semantic + Vector)

La combinación más poderosa: ejecuta búsqueda vectorial + palabras clave, y luego aplica reranking semántico. Proporciona los resultados más relevantes combinando todas las técnicas. Ideal para RAG en producción.

In [34]:
# Semantic Hybrid Search - La mejor combinación (Semantic + Vector)
print("\n" + "="*80)
print("4. SEMANTIC HYBRID SEARCH (Semantic + Vector)")
print("="*80)

# Definir SEMANTIC_CONFIG localmente
SEMANTIC_CONFIG = None

try:
    if SEMANTIC_CONFIG:
        results = client.search(
            search_text="cómo funcionan los embeddings",
            query_type="semantic",
            semantic_configuration_name=SEMANTIC_CONFIG,
            top=5
        )
    else:
        print("No hay configuración semántica disponible. Usando búsqueda simple.")
        results = client.search(
            search_text="cómo funcionan los embeddings",
            query_type="simple",
            top=5
        )
    
    print(f"\nQuery: 'cómo funcionan los embeddings'\n")
    for i, result in enumerate(results, 1):
        print(f"Resultado {i} | Score: {result['@search.score']:.4f}")
        if hasattr(result, '@search.reranker_score') and result.get('@search.reranker_score'):
            print(f"  Reranker Score: {result['@search.reranker_score']:.4f}")
        for key, value in result.items():
            if not key.startswith('@search'):
                value_preview = str(value)[:80]
                print(f"  {key}: {value_preview}")
        print()
        
except Exception as e:
    print(f" Error en Semantic Hybrid Search: {e}")

print("="*80)


4. SEMANTIC HYBRID SEARCH (Semantic + Vector)
No hay configuración semántica disponible. Usando búsqueda simple.

Query: 'cómo funcionan los embeddings'

Resultado 1 | Score: 0.4934
  title: Manual de Beneficios.pdf
  chunk_id: c184a177c51d_aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5
  text_vector: [-0.008553303, -0.003125764, 0.016297784, -0.04928428, -0.014545334, 0.015556362
  chunk: Manual de Beneficios y Bienestar del Empleado

Recursos Humanos - CloudTech Solu
  parent_id: aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5uZXQvYWxtYWNl

Resultado 2 | Score: 0.2525
  title: Visión General de la Empresa.pdf
  chunk_id: c184a177c51d_aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9iLmNvcmUud2luZG93cy5
  text_vector: [-0.014599704, -0.01998276, 0.014185622, -0.016589966, 0.02145208, 0.014265767, 
  chunk: CloudTech Solutions: Visión General y Cultura Corporativa

2024

Departamento de
  parent_id: aHR0cHM6Ly9hbG1hY2VuYW1pZW50b2VtYmVkaW5ncy5ibG9